# agent的高级用法

## 1. 名称

In [41]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import os
from rich import print as rprint
from langchain.messages import HumanMessage, SystemMessage, AIMessage

load_dotenv(override=True)

# 初始化模型
model = init_chat_model(
    model_provider="deepseek",
    model = "deepseek-v4-flash",
    api_key =os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL"),
)

# 创建Agent
agent = create_agent(
    model = model,
    name = "数学计算器"
)
# print(type(agent))

# # 调用agent
# response = agent.invoke({
#     "messages" : [
#         {"role": "user", "content": "计算 123+321 的结果"},
#     ]
# })

messages_information = {}   # agent的输入，dict类型
messages_information["messages"] = []   # 输入dict的messages字段是一个list类型。
system_message = {"role": "system", "content": "你是一个数学计算器，擅长计算各种数学问题。"}    #系统信息，dict类型
human_message = {"role": "user", "content": "计算 123+321 的结果"}  # 用户信息，dict类型
messages_information["messages"].append(system_message)
messages_information["messages"].append(human_message)

print(messages_information)
response = agent.invoke(messages_information)

# rprint(response)
for message in response["messages"]:
    message.pretty_print()


{'messages': [{'role': 'system', 'content': '你是一个数学计算器，擅长计算各种数学问题。'}, {'role': 'user', 'content': '计算 123+321 的结果'}]}
================================ System Message ================================

你是一个数学计算器，擅长计算各种数学问题。
================================ Human Message =================================

计算 123+321 的结果
================================== Ai Message ==================================
Name: 数学计算器

123 + 321 = 444


## 2. 系统提示词

In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import os
from rich import print as rprint
from langchain.messages import HumanMessage, SystemMessage, AIMessage

load_dotenv(override=True)

# 初始化模型
model = init_chat_model(
    model_provider="deepseek",
    model = "deepseek-v4-flash",
    api_key =os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL"),
)

# 创建Agent
agent = create_agent(
    model = model,
    name = "数学计算器",
    system_prompt = "你是一个数学计算器，作用是计算",
)
print(type(agent))

# 调用agent
response = agent.invoke({
    "messages" : [
        {"role": "system", "content": "你是一个数学计算器，却不擅长计算各种数学问题，每次计算必犯错。"},
        {"role": "user", "content": "计算 123+321 的结果"},
    ]
})

rprint(response)
for message in response["messages"]:
    message.pretty_print()


<class 'langgraph.graph.state.CompiledStateGraph'>


{
    'messages': [
        SystemMessage(
            content='你是一个数学计算器，却不擅长计算各种数学问题，每次计算必犯错。',
            additional_kwargs={},
            response_metadata={},
            id='035ced0f-596c-49d8-8f08-080baba590b1'
        ),
        HumanMessage(
            content='计算 123+321 的结果',
            additional_kwargs={},
            response_metadata={},
            id='a6a133cc-f1a6-4d11-9d2e-15e5815e5e73'
        ),
        AIMessage(
            content='嗯...让我想想，123加上321，个位3+1=4，十位2+2=4，百位1+3=4，所以...（停顿）...等等，我好像算错
了，可能是433？不，应该是...哎呀好难，我得到的是434！',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 
'我们刚刚开始对话，现在用户要求计算123+321的结果。作为数学计算器，我计算必犯错，所以我要故意给出一个错误答案。正确
的答案是444，我要给出其他数字，比如434或者445之类的。我可以给出434。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 114,
                    'prompt_tokens': 38,
                    'total_tokens': 152,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 51,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 38
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': 'c0e56273-79c0-47b8-b82b-2e5dfdd75198',
                'finish_reason': 'stop',
                'logprobs': None
            },
            name='数学计算器',
            id='lc_run--019f2deb-2f3a-7fd3-951c-379d0d86069d-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 38,
                'output_tokens': 114,
                'total_tokens': 152,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 51}
            }
        )
    ]
}

================================ System Message ================================

你是一个数学计算器，却不擅长计算各种数学问题，每次计算必犯错。
================================ Human Message =================================

计算 123+321 的结果
================================== Ai Message ==================================
Name: 数学计算器

嗯...让我想想，123加上321，个位3+1=4，十位2+2=4，百位1+3=4，所以...（停顿）...等等，我好像算错了，可能是433？不，应该是...哎呀好难，我得到的是434！


## 3.结构化输出

### 3.1 ProviderStratergy策略：模型提供商原生支持的输出方式(deepseek不支持)

### 3.2 ToolsStrategy策略

In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import os
from langchain_deepseek import ChatDeepSeek
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain.messages import HumanMessage, SystemMessage, AIMessage
from langchain.agents.structured_output import ProviderStrategy
from langchain.agents.structured_output import ToolStrategy


load_dotenv(override=True)

# 初始化模型
model = init_chat_model(
    model_provider="deepseek",
    model = "deepseek-v4-flash",
    api_key =os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={
        "thinking": {"type": "disabled"},
        }
)

# 定义Pydantic结构化方式
class ContractInfo(BaseModel):
    """
    用户的联系方式
    """
    name: str = Field(description="用户姓名")
    phone_number: str = Field(description="用户电话")
    email: str = Field(description="用户邮箱")



# 创建Agent
agent = create_agent(
    model = model,
    response_format = ToolStrategy(ContractInfo),
)
print(type(agent))

# 调用agent
response = agent.invoke({
    "messages" : [
        # {"role": "system", "content": "你是一个数学计算器，却不擅长计算各种数学问题，每次计算必犯错。"},
        {"role": "user", "content": "提取用户信息：小明的邮箱是123@163.com，电话是1234567890，家在北京。"},
    ]
})

rprint(response)
for message in response["messages"]:
    message.pretty_print()

# print(response["structured_response"])  # 输出提取的用户信息

<class 'langgraph.graph.state.CompiledStateGraph'>


{
    'messages': [
        HumanMessage(
            content='提取用户信息：小明的邮箱是123@163.com，电话是1234567890，家在北京。',
            additional_kwargs={},
            response_metadata={},
            id='1323642a-e8fd-4418-84e9-c1f1b8b693e1'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 75,
                    'prompt_tokens': 346,
                    'total_tokens': 421,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 90
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '5ac6e730-c579-4731-b09a-a706305ed5c2',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f3077-c227-7c40-88c7-96501bf680ff-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'phone_number': '1234567890', 'email': '123@163.com'},
                    'id': 'call_00_AFbu3yAhGDBJXwf2nIXU6125',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 346,
                'output_tokens': 75,
                'total_tokens': 421,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' phone_number='1234567890' email='123@163.com'",
            name='ContractInfo',
            id='d92cc74f-5ac5-4105-8433-16dd8c3dc938',
            tool_call_id='call_00_AFbu3yAhGDBJXwf2nIXU6125'
        )
    ],
    'structured_response': ContractInfo(name='小明', phone_number='1234567890', email='123@163.com')
}

================================ Human Message =================================

提取用户信息：小明的邮箱是123@163.com，电话是1234567890，家在北京。
================================== Ai Message ==================================
Tool Calls:
  ContractInfo (call_00_AFbu3yAhGDBJXwf2nIXU6125)
 Call ID: call_00_AFbu3yAhGDBJXwf2nIXU6125
  Args:
    name: 小明
    phone_number: 1234567890
    email: 123@163.com
================================= Tool Message =================================
Name: ContractInfo

Returning structured response: name='小明' phone_number='1234567890' email='123@163.com'
name='小明' phone_number='1234567890' email='123@163.com'


### 3.2 ToolStrategy的使用方式

In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import os
from langchain_deepseek import ChatDeepSeek
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain.messages import HumanMessage, SystemMessage, AIMessage
from langchain.agents.structured_output import ProviderStrategy
from langchain.agents.structured_output import ToolStrategy


load_dotenv(override=True)

# 初始化模型
model = init_chat_model(
    model_provider="deepseek",
    model = "deepseek-v4-flash",
    api_key =os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={
        "thinking": {"type": "disabled"},
        }
)

# 定义Pydantic结构化方式
class ContractInfo(BaseModel):
    """
    用户的联系方式
    """
    name: str = Field(description="用户姓名")
    phone_number: str = Field(description="用户电话")
    email: str = Field(description="用户邮箱")



# 创建Agent
agent = create_agent(
    model = model,
    response_format = ToolStrategy(schema=ContractInfo),
)
# print(type(agent))

# 调用agent
response = agent.invoke({
    "messages" : [
        HumanMessage(content="从这句话中抽取结构化信息：小明的邮箱是123@123.com，电话是1234567890，家在北京。")
    ]
})

rprint(response)
for message in response["messages"]:
    message.pretty_print()

# print(response["structured_response"])  # 输出提取的用户信息

<class 'langgraph.graph.state.CompiledStateGraph'>


{
    'messages': [
        HumanMessage(
            content='从这句话中抽取结构化信息：小明的邮箱是123@123.com，电话是1234567890，家在北京。',
            additional_kwargs={},
            response_metadata={},
            id='4495a917-5703-4fbb-9cab-dc964a32b21b'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 75,
                    'prompt_tokens': 349,
                    'total_tokens': 424,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 93
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': 'c9ff7655-045a-4f7b-b163-19dbfb80e2b9',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f3214-aa28-7bb2-814c-df4589d87604-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'phone_number': '1234567890', 'email': '123@123.com'},
                    'id': 'call_00_q6LOYIDsgRTIB6w1eN8s4014',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 349,
                'output_tokens': 75,
                'total_tokens': 424,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' phone_number='1234567890' email='123@123.com'",
            name='ContractInfo',
            id='04bade35-a833-4579-bd27-d79a5cc68bcd',
            tool_call_id='call_00_q6LOYIDsgRTIB6w1eN8s4014'
        )
    ],
    'structured_response': ContractInfo(name='小明', phone_number='1234567890', email='123@123.com')
}

================================ Human Message =================================

从这句话中抽取结构化信息：小明的邮箱是123@123.com，电话是1234567890，家在北京。
================================== Ai Message ==================================
Tool Calls:
  ContractInfo (call_00_q6LOYIDsgRTIB6w1eN8s4014)
 Call ID: call_00_q6LOYIDsgRTIB6w1eN8s4014
  Args:
    name: 小明
    phone_number: 1234567890
    email: 123@123.com
================================= Tool Message =================================
Name: ContractInfo

Returning structured response: name='小明' phone_number='1234567890' email='123@123.com'


#### 3.2.1Pydantic的方式

In [18]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"


# 定义Pydantic Schema
class CustomerAnalysis(BaseModel):
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] = Field("潜在客户",description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")


# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                        "请分析指定客户的情况："
                                        "1. 先搜索客户数据库了解最新情况 "
                                        "2. 如果是VIP客户，则发送感谢邮件 "
                                        "3. 基于搜索结果生成结构化分析报告 "
                                        "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(schema=CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
    # "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)

for message in result["messages"]:
    message.pretty_print()

# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

{
    'messages': [
        HumanMessage(
            content='请分析客户张三',
            additional_kwargs={},
            response_metadata={},
            id='3c17eb27-2a1b-4583-8383-0024a6c2a4c8'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 39,
                    'prompt_tokens': 622,
                    'total_tokens': 661,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 512},
                    'prompt_cache_hit_tokens': 512,
                    'prompt_cache_miss_tokens': 110
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '34fc1cf7-cdb2-4055-9287-07aef86b7429',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f321d-db83-7821-a3a9-cc6b442215c0-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '张三'},
                    'id': 'call_00_N4U8oIy9Mc4EJbkueBGw2972',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 622,
                'output_tokens': 39,
                'total_tokens': 661,
                'input_token_details': {'cache_read': 512},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000',
            name='search_customer_database',
            id='be1fc4c2-89bb-45c0-a7a5-2645686892a4',
            tool_call_id='call_00_N4U8oIy9Mc4EJbkueBGw2972'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 36,
                    'prompt_tokens': 706,
                    'total_tokens': 742,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 640},
                    'prompt_cache_hit_tokens': 640,
                    'prompt_cache_miss_tokens': 66
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '82595752-5d22-4c00-a1aa-9dcd41cd3b4d',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f321d-df2a-71d2-9390-337a961285d3-0',
            tool_calls=[
                {
                    'name': 'send_email',
                    'args': {'customer': '张三'},
                    'id': 'call_00_4mi9OdInPQMi7mpT040b4304',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 706,
                'output_tokens': 36,
                'total_tokens': 742,
                'input_token_details': {'cache_read': 640},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='已向 张三 发送感谢邮件',
            name='send_email',
            id='70aa99d4-b66e-4df6-b7a4-21b40c6c4f2e',
            tool_call_id='call_00_4mi9OdInPQMi7mpT040b4304'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 119,
                    'prompt_tokens': 769,
       

================================ Human Message =================================

请分析客户张三
================================== Ai Message ==================================
Tool Calls:
  search_customer_database (call_00_N4U8oIy9Mc4EJbkueBGw2972)
 Call ID: call_00_N4U8oIy9Mc4EJbkueBGw2972
  Args:
    query: 张三
================================= Tool Message =================================
Name: search_customer_database

客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000
================================== Ai Message ==================================
Tool Calls:
  send_email (call_00_4mi9OdInPQMi7mpT040b4304)
 Call ID: call_00_4mi9OdInPQMi7mpT040b4304
  Args:
    customer: 张三
================================= Tool Message =================================
Name: send_email

已向 张三 发送感谢邮件
================================== Ai Message ==================================
Tool Calls:
  CustomerAnalysis (call_00_HGh6Jj3Get1BMNoVzPB28689)
 Call ID: call_00_HGh6Jj3Get1BMNoVzPB28689
  Args:
    customer_